# Project 60 — Local Browser Task Agent
## Task Planning → DOM Simulation → Validation

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Browser Action Schema

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

class BrowserAction(BaseModel):
    step: int
    action: str = Field(description="navigate, click, type, scroll, wait, extract, screenshot")
    selector: str = Field(description="CSS selector or URL")
    value: str = Field(default="", description="Text to type or expected value")
    wait_ms: int = Field(default=0, description="Milliseconds to wait after action")
    explanation: str

class BrowserPlan(BaseModel):
    task: str
    url: str
    actions: list[BrowserAction]
    expected_result: str
    error_handling: list[str]
    estimated_seconds: int

planner = llm.with_structured_output(BrowserPlan)
print("Browser task planner ready!")

## Step 2 — Generate Task Plans

In [ ]:
tasks = [
    "Log into a web application, navigate to reports, download the latest CSV",
    "Search an e-commerce site for 'wireless keyboard', filter by price under $50, add the top result to cart",
    "Fill out a job application form with name, email, upload resume, and submit",
    "Navigate to a dashboard, take a screenshot of the revenue chart, extract the total value",
]

plans = []
for task in tasks:
    plan = planner.invoke(f"Create a browser automation plan: {task}")
    plans.append(plan)
    print(f"\n{'='*60}")
    print(f"Task: {plan.task}")
    print(f"URL: {plan.url}")
    print(f"Steps: {len(plan.actions)} | Est: {plan.estimated_seconds}s")
    for a in plan.actions:
        print(f"  {a.step}. [{a.action:<10}] {a.selector}")
        if a.value:
            print(f"     Value: {a.value}")
        print(f"     {a.explanation}")

## Step 3 — DOM Simulation & Validation

In [ ]:
# Simulate a simple DOM for testing
mock_dom = {
    "pages": {
        "/login": {
            "elements": {
                "#email": {"type": "input", "value": ""},
                "#password": {"type": "input", "value": ""},
                "#login-btn": {"type": "button", "text": "Sign In"},
            }
        },
        "/dashboard": {
            "elements": {
                "#revenue-chart": {"type": "div", "text": "Revenue: $1.2M"},
                "#download-btn": {"type": "button", "text": "Download Report"},
                ".nav-reports": {"type": "link", "href": "/reports"},
            }
        },
    }
}

class SimResult(BaseModel):
    success: bool
    steps_executed: int
    steps_failed: int
    final_state: str
    errors: list[str]

simulator = llm.with_structured_output(SimResult)

for plan in plans[:2]:
    result = simulator.invoke(
        f"Simulate this browser plan against the mock DOM.\n\n"
        f"Mock DOM: {json.dumps(mock_dom, indent=2)}\n\n"
        f"Plan: {json.dumps([a.model_dump() for a in plan.actions], indent=2)}"
    )
    print(f"\nTask: {plan.task[:50]}...")
    print(f"  Success: {result.success}")
    print(f"  Executed: {result.steps_executed}/{result.steps_executed + result.steps_failed}")
    if result.errors:
        for e in result.errors:
            print(f"  ✗ {e}")

## Step 4 — Generate Playwright Code

In [ ]:
codegen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert this browser automation plan into Playwright Python code. "
     "Include proper waits, error handling, and assertions."),
    ("human", "Plan:\n{plan}")
])
codegen_chain = codegen_prompt | llm | StrOutputParser()

for plan in plans[:2]:
    plan_json = json.dumps({
        "task": plan.task,
        "url": plan.url,
        "actions": [a.model_dump() for a in plan.actions],
    })
    playwright_code = codegen_chain.invoke({"plan": plan_json})
    print(f"\n{'='*60}")
    print(f"PLAYWRIGHT CODE — {plan.task[:40]}...")
    print("=" * 60)
    print(playwright_code[:600])

## What You Learned
- **Browser task decomposition** into atomic actions
- **DOM simulation** for plan validation
- **Playwright code generation** from plans
- **Error handling** for robust automation